# 139 — Document Vision Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/139-doc-vision-agent/doc_vision_agent_workbook.ipynb)

**What you'll learn:**
- How to rasterize PDF pages to PNG with `pypdfium2` and why DPI matters for token cost
- How to encode images as base64 for the OpenAI vision API
- How to write schema prompts that produce reliable JSON from a vision model
- How to select pages strategically (all / first-N / content-rich)
- How to implement the extract-then-combine pattern for multi-page documents
- How to wire a two-node LangGraph workflow (load → extract) and run it against a real PDF

**Source:** `examples/139-doc-vision-agent/`  
**Requirements:** `openai python-dotenv langgraph pypdfium2 pillow httpx`

| Concept | Why it matters |
|---------|----------------|
| `pypdfium2` → PIL → PNG | PDFs are vector programs; the vision API needs raster pixels |
| DPI and scale factor | Scale controls token cost — 2× is the sweet spot for body text |
| Schema prompt | Telling the model the exact JSON shape is faster than free-form extraction |
| `strip("\`\`\`json")` cleanup | LLMs often wrap JSON in markdown fences — strip before `json.loads()` |
| Page selection | Processing only content-rich pages cuts cost without losing signal |
| Extract-then-combine | Per-page extractions are cheap to parallelize and easy to merge |

In [ ]:
%pip install -q openai python-dotenv langgraph pypdfium2 pillow httpx

---
## Part A — Core Concepts (no API key required)

The cells in Part A are pure Python. Run them anywhere — Colab, local, or CI — without an OpenAI key.  
They build intuition for the rendering pipeline, cost model, and extraction patterns before you spend money on the live API.

### PDF rendering approaches

Three libraries can convert PDF pages to images. Each has a different profile:

| Library | Speed | Text accuracy | Output | Best for |
|---------|-------|---------------|--------|----------|
| **pypdfium2** | Fast (C library) | Excellent | PIL Image / bytes | Production pipelines, headless servers |
| **pdf2image** | Medium (Poppler wrapper) | Excellent | PIL Image list | Systems where Poppler is already installed |
| **pdfplumber** | Slow | Good for text extraction; no pixel output | Text / table objects | Text-only extraction without vision |

`pypdfium2` is the right choice here because:
- It ships as a self-contained wheel (no system dependency on Poppler)
- It exposes `render(scale=N)` directly — no intermediate file on disk
- The output is a PIL `Image`, which is exactly what we need for base64 encoding

Use `pdf2image` if your deployment already has Poppler.  
Use `pdfplumber` only when you want raw text or table extraction without invoking a vision model.

### DPI vs token cost

PDF coordinate space is 72 points per inch. The `scale` argument in `pypdfium2` is a multiplier on that baseline:

| `scale` | Effective DPI | Typical PNG size (A4 page) | Vision tokens (gpt-4o) | Cost per page (gpt-4o-mini) |
|---------|--------------|---------------------------|------------------------|------------------------------|
| 1 | 72 DPI | ~100–200 KB | ~255 tokens | ~$0.0001 |
| 2 | 144 DPI | ~300–600 KB | ~765 tokens | ~$0.0003 |
| 3 | 216 DPI | ~700 KB–1.5 MB | ~1 530 tokens | ~$0.0006 |
| 4 | 288 DPI | ~1–3 MB | ~2 550 tokens | ~$0.0010 |

**Rule of thumb:** `scale=2` (144 DPI) is the right default for text-heavy documents.  
It keeps body text legible for the model while staying well below the image size threshold where cost jumps.  
Use `scale=3` only for documents with small footnotes or dense tables you need to read accurately.

> Token counts above are approximate. gpt-4o tiles images into 512×512 patches and charges ~85 tokens per tile plus a base cost. Actual cost depends on image dimensions.

In [ ]:
# Create a synthetic PIL image (no PDF needed), encode it to base64, show byte size
# This illustrates the encode → base64 step independent of pypdfium2

import base64
import io
from PIL import Image, ImageDraw, ImageFont

# Build a minimal synthetic document page
img = Image.new("RGB", (800, 600), color=(255, 255, 255))
draw = ImageDraw.Draw(img)

draw.rectangle([40, 40, 760, 80], fill=(30, 30, 120))
draw.text((50, 50), "Sample Document  —  Page 1", fill=(255, 255, 255))
draw.text((50, 110), "Section 1: Introduction", fill=(0, 0, 0))
draw.text((50, 150), "Key figures: 42%, 1 024, $3.7 billion", fill=(0, 0, 0))
draw.text((50, 190), "Date: 2024-01-15", fill=(0, 0, 0))
draw.rectangle([40, 240, 760, 320], outline=(200, 200, 200), width=2)
draw.text((50, 260), "Table 1: Summary statistics", fill=(80, 80, 80))

# Encode to PNG base64 — same operation used in src/tools.py
buf = io.BytesIO()
img.save(buf, format="PNG")
png_bytes = buf.getvalue()
b64_synthetic = base64.b64encode(png_bytes).decode()

print(f"Image size      : {img.width} x {img.height} px")
print(f"PNG byte size   : {len(png_bytes):,} bytes  ({len(png_bytes)/1024:.1f} KB)")
print(f"Base64 length   : {len(b64_synthetic):,} chars")
print(f"Overhead factor : {len(b64_synthetic)/len(png_bytes):.2f}x  (base64 is always ~1.33x the binary size)")
print(f"\nFirst 80 chars  : {b64_synthetic[:80]}...")

### Page selection strategies

For a 100-page PDF you rarely need all 100 pages. Common strategies:

| Strategy | When to use | How to implement |
|----------|-------------|------------------|
| **All pages** | Short docs (<10 pages), compliance extraction, audits | `range(len(doc))` |
| **First N pages** | Executive summaries, cover + table of contents | `range(min(N, len(doc)))` |
| **Keyword-guided** | Known section titles (e.g. "Financial Summary") | Render each page, OCR or text-extract, filter by keyword |
| **Structure-based** | Content-rich pages (images, tables, dense text) | Filter by rendered image file size (larger = more content) |
| **Odd pages only** | Two-column documents where even pages mirror odd | `range(0, len(doc), 2)` |

The structure-based approach is particularly useful: a blank or near-blank page renders to a small PNG (~15–30 KB), while a dense page with charts and tables renders to 200–600 KB. You can use file size as a cheap proxy for content density without calling the vision API at all.

### The extract-then-combine pattern

The two-node LangGraph workflow in `src/workflow.py` follows a classic map-reduce shape:

```
PDF source
    │
    ▼
[ load_node ]
    │  fetch bytes, count pages
    ▼
[ extract_node ]
    │  for each page:
    │      render PNG  →  base64
    │      vision API call  →  JSON
    │  collect list of JSON objects
    ▼
results: [ {page:1, ...}, {page:2, ...}, ... ]
```

The final "combine" step is just iterating over `state['results']`.  
For more advanced use cases you can add a third `summarize_node` that sends the list of per-page JSON objects to a text model (no image needed) for a final synthesis:

```
[ extract_node ]  →  list of page JSONs
    ▼
[ summarize_node ]  →  single document-level summary
```

This keeps the vision API cost bounded (one call per page) and moves the cheap synthesis step to a text-only call.

In [ ]:
# Mock extract_page() that returns a fixed string per page
# Then combine three page extractions with a separator

def mock_extract_page(page_index: int) -> str:
    """Simulate the text a vision model might return for a page."""
    pages = [
        "Title: WCAG 2.1 Specification. Date: June 2018. Summary: Defines accessibility guidelines for web content.",
        "Section: Table of Contents. Covers 4 principles: Perceivable, Operable, Understandable, Robust.",
        "Success Criterion 1.1.1 Non-text Content (Level A). Key number: 1.1.1. Requires text alternatives.",
    ]
    return pages[page_index % len(pages)]


# Run mock extraction for pages 1–3 and combine
separator = "\n---\n"
findings = [mock_extract_page(i) for i in range(3)]
combined = separator.join(f"[Page {i+1}] {text}" for i, text in enumerate(findings))

print("Per-page extractions:")
for i, f in enumerate(findings):
    print(f"  Page {i+1}: {f[:70]}...")

print("\nCombined output:")
print(combined)

In [ ]:
# Cost estimation function: given n_pages and scale, estimate total tokens and cost
# Uses the gpt-4o tile model: base 85 tokens + 170 tokens per 512x512 tile

def estimate_vision_cost(
    n_pages: int,
    scale: float = 2.0,
    page_width_pt: float = 595,   # A4 width in PDF points
    page_height_pt: float = 842,  # A4 height in PDF points
    price_per_1k_tokens: float = 0.00015,  # gpt-4o-mini input price
) -> dict:
    """
    Estimate token count and cost for rendering n_pages at a given scale.

    gpt-4o vision pricing (approximate):
    - Low-detail mode: 85 tokens flat per image
    - High-detail mode: 85 + 170 per 512x512 tile (tiles computed after resize to 2048 on long side)
    """
    # Rendered pixel dimensions
    px_w = int(page_width_pt * scale)
    px_h = int(page_height_pt * scale)

    # High-detail tile count (gpt-4o resizes so longest side <= 2048, then 512px tiles)
    max_side = max(px_w, px_h)
    if max_side > 2048:
        ratio = 2048 / max_side
        px_w = int(px_w * ratio)
        px_h = int(px_h * ratio)

    import math
    tiles_w = math.ceil(px_w / 512)
    tiles_h = math.ceil(px_h / 512)
    n_tiles = tiles_w * tiles_h
    tokens_per_page = 85 + 170 * n_tiles

    total_tokens = tokens_per_page * n_pages
    total_cost = (total_tokens / 1000) * price_per_1k_tokens

    return {
        "scale": scale,
        "rendered_px": f"{int(page_width_pt*scale)} x {int(page_height_pt*scale)}",
        "tiles_per_page": n_tiles,
        "tokens_per_page": tokens_per_page,
        "n_pages": n_pages,
        "total_tokens": total_tokens,
        "estimated_cost_usd": round(total_cost, 4),
    }


print("Cost estimates for a 60-page A4 PDF at different scales (gpt-4o-mini pricing):")
print(f"{'Scale':>6}  {'Rendered (px)':>18}  {'Tiles':>5}  {'Tok/pg':>6}  {'Total tok':>9}  {'Cost USD':>9}")
print("-" * 65)
for s in [1, 2, 3, 4]:
    r = estimate_vision_cost(60, scale=s)
    print(f"  {r['scale']:>4}  {r['rendered_px']:>18}  {r['tiles_per_page']:>5}  "
          f"{r['tokens_per_page']:>6}  {r['total_tokens']:>9}  ${r['estimated_cost_usd']:>8.4f}")

### Extraction output formats

You have three main choices for what the model returns per page:

| Format | Example | Pros | Cons |
|--------|---------|------|------|
| **Plain text** | `"Page 1 introduces WCAG 2.1..."` | Simple to prompt, easy to read | Hard to parse downstream, no structure |
| **Markdown** | `# Title\n**Key number:** 1.1.1` | Human-readable, preserves hierarchy | Inconsistent across models, tricky to parse |
| **JSON** | `{"title": "...", "key_numbers": [...]}` | Machine-parseable, schema-enforced | Slightly more tokens in prompt; fence-stripping needed |

**Use JSON when** you will aggregate, filter, or store the results.  
**Use plain text or markdown when** the next step is another LLM call (e.g., a summarization agent that reads all page outputs).

The `strip("\`\`\`json")` pattern in `src/tools.py` handles the most common LLM response noise — the model wraps its JSON in a markdown code block. Always strip before calling `json.loads()`.

In [ ]:
# Two extraction prompts side by side: plain text vs structured JSON
# These are the actual strings you'd pass to the vision API as the text message

PLAIN_TEXT_PROMPT = """\
Read this document page and write a 2–3 sentence summary of the main content.
Include any important numbers, dates, or proper nouns you see.
Respond with plain text only."""

JSON_PROMPT = """\
Extract the following fields from this document page and return ONLY valid JSON.
Do not wrap the JSON in markdown fences.

{
  "heading": "the main heading or title visible on this page, or null",
  "body_text": "a 1-2 sentence summary of the body content",
  "tables": ["description of each table if any, e.g. 'Table 1: Sales by region'"],
  "images_described": ["description of each figure/chart/image if any"]
}"""

print("=" * 60)
print("PLAIN TEXT PROMPT")
print("=" * 60)
print(PLAIN_TEXT_PROMPT)

print()
print("=" * 60)
print("STRUCTURED JSON PROMPT")
print("=" * 60)
print(JSON_PROMPT)

print()
print("Key difference: the JSON prompt names explicit fields.")
print("This reduces hallucination of field names and makes downstream parsing trivial.")

### OCR accuracy vs vision model accuracy

Classic OCR (Tesseract, AWS Textract, Google Document AI) and vision models solve the same problem differently:

| Dimension | Classic OCR | Vision model (GPT-4o) |
|-----------|-------------|------------------------|
| **Raw character accuracy** | Very high on clean scans (>99%) | High but not guaranteed character-perfect |
| **Semantic understanding** | None — just characters | Understands layout, headings, table structure |
| **Handwriting** | Limited (Tesseract poor, cloud OCR better) | Good on clear handwriting |
| **Mixed language** | Requires language hints | Handles mixed-language pages natively |
| **Table extraction** | Needs post-processing | Can describe table structure in JSON |
| **Cost** | $0.0015–$0.015 per page (cloud OCR) | $0.0001–$0.001 per page (gpt-4o-mini) |
| **Latency** | Fast (100–300ms) | Slower (1–3s per page) |

**When to use each:**
- **Classic OCR** → you need character-perfect verbatim text extraction (legal documents, invoices with exact amounts)
- **Vision model** → you need structured extraction, semantic understanding, or the document has complex layout (annual reports, research papers, mixed tables and prose)

### Handling multi-column PDFs

Academic papers, newspapers, and brochures often use two- or three-column layouts. This is a known challenge:

- **Classic OCR** reads left-to-right, top-to-bottom — it mixes column 1 row 5 with column 2 row 5 mid-sentence
- **Vision models** generally understand column layout intuitively, but you can prompt-guide them:

```python
multi_column_prompt = """
This page uses a two-column layout.
Read the LEFT column top-to-bottom first, then the RIGHT column top-to-bottom.
Extract as JSON:
{
  "left_column_summary": "...",
  "right_column_summary": "...",
  "shared_heading": "... or null"
}
"""
```

Explicit column instruction in the prompt reliably fixes reading order issues for two-column academic layouts.

---
### Exercise 1 — Structured extraction dict

Modify `mock_extract_page()` to return a Python `dict` with structured fields instead of a plain string.  
The dict should have keys: `heading`, `summary` (one sentence), and `key_facts` (list of strings).

Then iterate over pages 0–2 and print the structured output.

In [ ]:
# Exercise 1: Modify mock_extract_page() to return a structured dict
# Keys required: heading (str), summary (str, one sentence), key_facts (list of str)

def mock_extract_page_structured(page_index: int) -> dict:
    # TODO: return a dict with keys: heading, summary, key_facts
    pass


# Test your implementation:
for i in range(3):
    result = mock_extract_page_structured(i)
    print(f"Page {i+1}: {result}")

In [ ]:
# Exercise 1 — Answer Key
# The function returns a dict directly rather than a JSON string.
# In the real workflow, extract_page() in src/tools.py returns json.loads(text),
# so the downstream code receives a dict either way.

import json

def mock_extract_page_structured_answer(page_index: int) -> dict:
    """
    Returns a structured dict mimicking the JSON a vision model would produce.
    In production, the model fills these fields from the actual page image.
    """
    pages = [
        {
            "heading": "Web Content Accessibility Guidelines (WCAG) 2.1",
            "summary": "This cover page introduces WCAG 2.1, published by W3C in June 2018.",
            "key_facts": ["Published: 5 June 2018", "Version: 2.1", "Publisher: W3C"]
        },
        {
            "heading": "Table of Contents",
            "summary": "The table of contents lists four core principles of web accessibility.",
            "key_facts": ["Principle 1: Perceivable", "Principle 2: Operable",
                          "Principle 3: Understandable", "Principle 4: Robust"]
        },
        {
            "heading": "Success Criterion 1.1.1 Non-text Content",
            "summary": "All non-text content must have a text alternative that serves the equivalent purpose.",
            "key_facts": ["Level: A", "Criterion: 1.1.1", "Applies to: images, controls, time-based media"]
        },
    ]
    return pages[page_index % len(pages)]


print("Answer Key output:")
for i in range(3):
    result = mock_extract_page_structured_answer(i)
    print(f"\nPage {i+1}:")
    print(json.dumps(result, indent=2))

---
### Exercise 2 — Selective page extraction

Write a `select_pages()` function that identifies content-rich page indices using rendered image size as a proxy.  
Return only the indices where the rendered PNG exceeds a given byte threshold (default: 50 000 bytes).

This lets you skip blank or near-blank pages without spending tokens on a vision API call.

In [ ]:
# Exercise 2: Write select_pages() that returns indices of content-rich pages
# Hint: render each page to a BytesIO buffer and check len(buf.getvalue())
# You'll need: import io, import pypdfium2 as pdfium

def select_pages(pdf_bytes: bytes, min_bytes: int = 50_000) -> list[int]:
    """
    Return page indices where the rendered PNG exceeds min_bytes.
    Use scale=1 for the size-check render to keep this step cheap.
    """
    # TODO: implement
    pass


# Smoke test against the WCAG PDF bytes fetched earlier (uses b64_page0 as a proxy)
# If you haven't run the PDF fetch cell yet, this will raise a NameError — that's fine.
try:
    rich_pages = select_pages(pdf_bytes)
    print(f"Content-rich pages (>{50_000} bytes): {rich_pages[:10]} ...")
    print(f"Total pages: {len(rich_pages)} / {len(pdfium.PdfDocument(pdf_bytes))}")
except NameError:
    print("Run the PDF fetch cell (Part A · Cell 2 equivalent) first to get pdf_bytes.")

In [ ]:
# Exercise 2 — Answer Key
# Key insight: render at scale=1 (not scale=2) for the size check.
# The low-resolution render is 4x cheaper (fewer pixels) but still reflects
# whether the page has content — blank pages are tiny at any scale.

import io
import pypdfium2 as pdfium

def select_pages_answer(pdf_bytes: bytes, min_bytes: int = 50_000, check_scale: float = 1.0) -> list[int]:
    """
    Return page indices where the rendered PNG exceeds min_bytes.

    Uses check_scale=1 (72 DPI) for the size check render — much cheaper than
    the scale=2 render used for the actual vision API call.
    """
    doc = pdfium.PdfDocument(pdf_bytes)
    rich_indices = []
    for i in range(len(doc)):
        page = doc[i]
        bitmap = page.render(scale=check_scale)
        img = bitmap.to_pil()
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        if len(buf.getvalue()) >= min_bytes:
            rich_indices.append(i)
    return rich_indices


# Test the answer key
try:
    rich = select_pages_answer(pdf_bytes, min_bytes=50_000)
    total_pages = len(pdfium.PdfDocument(pdf_bytes))
    skipped = total_pages - len(rich)
    print(f"Content-rich page indices (first 10): {rich[:10]}")
    print(f"Total pages   : {total_pages}")
    print(f"Rich pages    : {len(rich)}")
    print(f"Skipped pages : {skipped}  (would have been blank API calls)")
except NameError:
    print("Run the PDF fetch cell first to populate pdf_bytes.")

### The full rendering pipeline

Below is the complete path from PDF bytes to base64 string, as implemented in `src/tools.py`.
Reading this helps you understand where each transformation happens and what you'd change if you needed JPEG instead of PNG, or a different DPI.

```
pdf_bytes: bytes
    └─► pypdfium2.PdfDocument(pdf_bytes)        # parse in-memory; no temp file
            └─► doc[page_index]                 # access one page object
                    └─► page.render(scale=2)    # rasterize at 144 DPI
                            └─► bitmap.to_pil() # → PIL Image (RGB)
                                    └─► img.save(buf, format="PNG")  # lossless
                                            └─► base64.b64encode(buf.getvalue())
                                                        └─► .decode()  # → str for JSON
```

**To switch to JPEG:** replace `format="PNG"` with `format="JPEG"` and add `quality=85`.  
Use JPEG for photograph-heavy documents; keep PNG for text and diagrams.

**To change DPI:** change `scale=2` — e.g., `scale=1` for a cheap 72-DPI check, `scale=3` for small footnotes.

**To process from a file path:** `pdf_bytes = open("report.pdf", "rb").read()` — same pipeline from there.

In [ ]:
# Part A · Render the first page of a public PDF to PNG
# No API key needed — just pypdfium2 and httpx

import base64
import io
import httpx
import pypdfium2 as pdfium

SAMPLE_PDF_URL = "https://www.w3.org/WAI/WCAG21/wcag21.pdf"

print(f"Fetching PDF from: {SAMPLE_PDF_URL}")
resp = httpx.get(SAMPLE_PDF_URL, timeout=30)
resp.raise_for_status()
pdf_bytes = resp.content

doc = pdfium.PdfDocument(pdf_bytes)
print(f"PDF loaded: {len(doc)} pages, {len(pdf_bytes)/1024:.1f} KB")

# Render page 0 at scale=2 (144 DPI)
page = doc[0]
width_pt, height_pt = page.get_width(), page.get_height()
bitmap = page.render(scale=2)
pil_img = bitmap.to_pil()

print(f"\nPage 0 PDF points   : {width_pt:.0f} x {height_pt:.0f} pt")
print(f"Rendered pixels (2x): {pil_img.width} x {pil_img.height} px")

# Encode to PNG base64
buf = io.BytesIO()
pil_img.save(buf, format="PNG")
b64_page0 = base64.b64encode(buf.getvalue()).decode()

print(f"PNG size            : {len(buf.getvalue())/1024:.1f} KB")
print(f"Base64 length       : {len(b64_page0):,} chars")
print(f"\nFirst 60 chars: {b64_page0[:60]}...")

### The schema prompt approach

Prompt engineering for structured output works by telling the model exactly what JSON shape to produce and instructing it to respond *only* with JSON.

```python
SCHEMA_PROMPT = """
Extract the following fields from this document page as JSON:
{
  "title": "string or null",
  "date": "string or null",
  "key_numbers": ["list of numeric values mentioned"],
  "summary": "one sentence summary"
}
"""
```

The text message appended to the image block is: `<schema_prompt>\n\nRespond with valid JSON only.`

**Why not use `response_format={"type": "json_object"}`?**  
That flag forces syntactically valid JSON but does *not* enforce the schema. With a clear schema in the prompt, the model follows the field names reliably.

**The fence-stripping pattern** handles common LLM output noise:
```python
text = resp.choices[0].message.content.strip()
text = text.strip("```json").strip("```")  # remove markdown fences
json.loads(text)
```

In [ ]:
# Mock extract_page: show what the LLM response JSON looks like, including fence variants
# Simulates the output of src/tools.py extract_page() without an API call

import json

SCHEMA_PROMPT = """Extract the following fields from this document page as JSON:
{
  "title": "string or null",
  "date": "string or null",
  "key_numbers": ["list of numeric values mentioned"],
  "summary": "one sentence summary"
}"""


def mock_llm_response(page_hint: str) -> str:
    """Simulate the raw LLM response — sometimes wrapped in markdown fences."""
    responses = {
        "cover": '{\n  "title": "WCAG 2.1",\n  "date": "5 June 2018",\n  "key_numbers": ["2.1"],\n  "summary": "Cover page of the W3C WCAG 2.1 specification."\n}',
        "toc":   '```json\n{\n  "title": "Table of Contents",\n  "date": null,\n  "key_numbers": ["1", "2", "3", "4"],\n  "summary": "Lists four principles: Perceivable, Operable, Understandable, Robust."\n}\n```',
        "body":  '  {"title": null, "date": null, "key_numbers": ["1.1.1", "4.5:1"], "summary": "Success criterion 1.1.1 requires text alternatives for non-text content."}  ',
    }
    return responses.get(page_hint, '{"title": null, "date": null, "key_numbers": [], "summary": "Empty page."}')


def mock_extract_page(b64_png: str, schema_prompt: str, page_hint: str = "cover") -> dict:
    """Mirrors src/tools.py extract_page() but uses a simulated response."""
    raw = mock_llm_response(page_hint)
    text = raw.strip().strip("```json").strip("```").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}


print("Simulating extract_page() on 3 different page types:\n")
for hint in ["cover", "toc", "body"]:
    result = mock_extract_page(b64_page0, SCHEMA_PROMPT, page_hint=hint)
    print(f"--- Page type: {hint} ---")
    print(json.dumps(result, indent=2))
    print()

print("Note: fence-stripping handles all three raw response formats correctly.")

---
## Part B — Live API Run

The cells below require `OPENAI_API_KEY` in your `.env` file (or environment).

The full workflow is a two-node LangGraph graph:
```
START → load_node → extract_node → END
          │                │
          │ fetch PDF,     │ loop over pages,
          │ count pages    │ call vision API per page
          ▼                ▼
        page_count       results list
```

**Cost warning:** The WCAG PDF has 60+ pages. Cell below only processes the first 3 pages. Adjust `MAX_PAGES` to process more.

In [ ]:
# Part B · Fail-fast setup — verify API key and connectivity before iterating pages

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

try:
    import pypdfium2  # noqa: F401
    print("pypdfium2 available.")
except ImportError:
    raise ImportError("pypdfium2 not installed. Run: pip install pypdfium2")

api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    raise EnvironmentError(
        "OPENAI_API_KEY not set.\n"
        "Add it to your .env file or set the environment variable, then re-run this cell."
    )

client = OpenAI(api_key=api_key)

try:
    client.models.list()
    print("Connected to OpenAI API.")
except Exception as e:
    raise ConnectionError(f"Cannot reach OpenAI API: {e}") from e

print(f"PDF source : {SAMPLE_PDF_URL}")
print(f"PDF pages  : {len(doc)}")
print("Ready to run extraction workflow.")

In [ ]:
# Part B · Run the LangGraph load → extract workflow
# Processes up to MAX_PAGES from the WCAG PDF to control cost

from langgraph.graph import StateGraph, END
from typing import TypedDict

MAX_PAGES = 3  # increase to process more pages


class DocState(TypedDict):
    pdf_source: str
    schema_prompt: str
    page_count: int
    results: list


def pdf_page_to_b64_live(pdf_bytes: bytes, page_index: int) -> str:
    doc_local = pdfium.PdfDocument(pdf_bytes)
    page = doc_local[page_index]
    bitmap = page.render(scale=2)
    img = bitmap.to_pil()
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


def extract_page_live(b64_png: str, schema_prompt: str, _client, model: str) -> dict:
    resp = _client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64_png}"}},
            {"type": "text", "text": schema_prompt + "\n\nRespond with valid JSON only."},
        ]}],
        max_tokens=1024,
    )
    text = resp.choices[0].message.content.strip().strip("```json").strip("```").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}


def load_node(state: DocState, config: dict) -> DocState:
    _pdf_bytes = httpx.get(state["pdf_source"], timeout=30).content
    _doc = pdfium.PdfDocument(_pdf_bytes)
    return {**state, "_pdf_bytes": _pdf_bytes, "page_count": min(MAX_PAGES, len(_doc))}


def extract_node(state: DocState, config: dict) -> DocState:
    _client = config["configurable"]["client"]
    model = config["configurable"].get("model", "gpt-4o-mini")
    _pdf_bytes = state["_pdf_bytes"]
    results = []
    for i in range(state["page_count"]):
        print(f"  Extracting page {i + 1}/{state['page_count']}...", end=" ")
        b64 = pdf_page_to_b64_live(_pdf_bytes, i)
        result = extract_page_live(b64, state["schema_prompt"], _client, model)
        result["page"] = i + 1
        results.append(result)
        print("done")
    return {**state, "results": results}


graph = StateGraph(DocState)
graph.add_node("load", load_node)
graph.add_node("extract", extract_node)
graph.set_entry_point("load")
graph.add_edge("load", "extract")
graph.add_edge("extract", END)
workflow = graph.compile()

cfg = {"configurable": {"client": client, "model": "gpt-4o-mini"}}

print(f"Running workflow on {SAMPLE_PDF_URL} (first {MAX_PAGES} pages)...\n")
output = workflow.invoke(
    {"pdf_source": SAMPLE_PDF_URL, "schema_prompt": SCHEMA_PROMPT, "page_count": 0, "results": []},
    config=cfg,
)
print(f"\nProcessed {output['page_count']} pages.")

In [ ]:
# Part B · Display per-page extracted results

print("=" * 60)
print("Extracted structured data per page:")
print("=" * 60)
for page_result in output["results"]:
    print(f"\nPage {page_result.get('page', '?')}:")
    print(json.dumps(page_result, indent=2))

print("\n" + "=" * 60)
print(f"Total pages processed : {output['page_count']}")
all_numbers = [n for r in output["results"] for n in r.get("key_numbers", [])]
print(f"All key numbers found : {all_numbers}")

---
## What's Next

- **GPT-4V for OCR** — OpenAI blog post on using vision models for document understanding: [platform.openai.com/docs/guides/vision](https://platform.openai.com/docs/guides/vision)
- **NOUGAT** — Meta's academic document OCR model (Blecher et al., 2023): handles LaTeX equations and two-column layouts: [arxiv.org/abs/2308.13418](https://arxiv.org/abs/2308.13418)
- **Example 138 — Vision Q&A Agent** — ask free-form questions about image content instead of extracting to a fixed schema: `examples/138-vision-qa-agent/`
- **Example 42 — Multimodal RAG** — embed page images alongside text chunks so you can retrieve by visual similarity: `examples/42-multimodal-rag/`